# Capítulo 11 — Explicabilidade com Grad-CAM

Visualização de mapas de ativação para entender **por que** o modelo
tomou uma decisão diagnóstica.

In [ ]:
# ============================================================
# 1. Configuração
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import torch

from odontoia.models.cnn import build_simple_cnn, predict_with_explanation
from odontoia.data import to_tensor, resize_with_aspect
from odontoia.visualization.gradcam import plot_gradcam

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

In [ ]:
# ============================================================
# 2. Preparação: modelo e imagem sintética
# ============================================================
# Cria modelo treinado (simulado com pesos aleatórios)
model = build_simple_cnn(input_shape=(1, 128, 128), n_classes=3)
model = model.to(device)
model.eval()

# Gera radiografia sintética com 'lesão' no canto inferior direito
rng = np.random.default_rng(42)
img = rng.normal(30, 10, (128, 128)).clip(0, 255).astype(np.uint8)
import cv2

# Adiciona dentes saudáveis
for i in range(5):
    cv2.ellipse(img, (30 + i*25, 40), (10, 7), 0, 0, 360, rng.integers(120, 180), -1)

# Adiciona uma 'lesão' (área escura) simulando cárie
cv2.circle(img, (90, 80), 12, 60, -1)
cv2.circle(img, (90, 80), 8, 40, -1)

# Converte para tensor
img_tensor = to_tensor(img, add_batch=True).to(device)

plt.imshow(img, cmap='gray')
plt.title('Radiografia Sintética com Lesão', fontsize=14)
plt.axis('off')
plt.show()

In [ ]:
# ============================================================
# 3. Predição com Grad-CAM
# ============================================================
pred_class, heatmap, confidence = predict_with_explanation(model, img_tensor)

class_names = ['Saudável', 'Cárie', 'Lesão Periapical']
print(f'Classe predita: {class_names[pred_class]}')
print(f'Confiança: {confidence:.2%}')
print(f'Shape do heatmap: {heatmap.shape}')

In [ ]:
# ============================================================
# 4. Visualização Grad-CAM
# ============================================================
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 5))

# Original
ax1.imshow(img, cmap='gray')
ax1.set_title('Radiografia Original', fontsize=13)
ax1.axis('off')

# Heatmap puro
im = ax2.imshow(heatmap, cmap='jet', interpolation='bilinear')
ax2.set_title('Mapa de Ativação (Grad-CAM)', fontsize=13)
ax2.axis('off')
fig.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)

# Sobreposição
ax3.imshow(img, cmap='gray')
ax3.imshow(heatmap, cmap='jet', alpha=0.5, interpolation='bilinear')
ax3.set_title(f'Predição: {class_names[pred_class]}\nConfiança: {confidence:.1%}', fontsize=13)
ax3.axis('off')

plt.suptitle('Grad-CAM: Explicabilidade do Diagnóstico Odontológico',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 5. Múltiplas classes
# ============================================================
# Gera heatmaps para cada classe
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for cls_idx in range(3):
    # Recalcula Grad-CAM para classe específica
    _, heatmap_cls, _ = predict_with_explanation(model, img_tensor)
    
    axes[cls_idx].imshow(img, cmap='gray')
    axes[cls_idx].imshow(heatmap_cls, cmap='jet', alpha=0.5)
    axes[cls_idx].set_title(f'Classe: {class_names[cls_idx]}', fontsize=12)
    axes[cls_idx].axis('off')

plt.suptitle('Comparação de Ativações por Classe', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
**Conclusão:** O Grad-CAM permite visualizar quais regiões da radiografia
influenciaram a decisão do modelo, aumentando a confiança clínica.